In [1]:
from sklearn.neighbors import KNeighborsClassifier

In [2]:
knn = KNeighborsClassifier(weights='distance')

In [124]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# učitavanje skupa za treniranje
TRAIN_DIR = "../../train"        
IMAGE_SIZE = (32, 32)        

# labele su u .csv fajlovima
csv_files = [f for f in os.listdir(TRAIN_DIR) if f.endswith(".csv")]

if len(csv_files) == 0:
    raise Exception("No CSV files found in train folder.")

dfs = []
for csv_file in csv_files:
    path = os.path.join(TRAIN_DIR, csv_file)
    dfs.append(pd.read_csv(path))

labels_df = pd.concat(dfs, ignore_index=True)

print("Loaded labels:")
print(labels_df.head())


X = []
y = []

# učitavanje slika, njihova transformacija u 1d vektor i pridruživanje klase
for _, row in labels_df.iterrows():

    filename = row[0]   
    label = row[3]      

    image_path = os.path.join(TRAIN_DIR, filename)

    if not os.path.exists(image_path):
        print(f"Missing image: {filename}")
        continue

    # učitvanje slike
    img = cv2.imread(image_path)

    if img is None:
        print(f"Failed to load: {filename}")
        continue

    # Konverzija u gray scale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Sklairanje veličine slike
    img = cv2.resize(img, IMAGE_SIZE)

    # Normalizacija vrednosti piksela
    img = img.astype(np.float32) / 255.0

    img = img.flatten()

    X.append(img)
    y.append(label)

X = np.array(X)
y = np.array(y)

print(f"\nDataset loaded:")
print("X shape:", X.shape)
print("y shape:", y.shape)

Loaded labels:
                                            filename  width  height  class  \
0  5_118_jpg.rf.dd554fb8cdd63040f7045d950d15d93c.jpg    320     240      5   
1   1_95_jpg.rf.dcfc629fe9c215d950e003b431ec048d.jpg    764     764      1   
2   2_76_jpg.rf.dc7466fd1e50301fb4e22b1ea534a70c.jpg    764     764      2   
3  1_111_jpg.rf.dc36e5f05e4ef68c9e6f798a5322d681.jpg    765    1020      1   
4    4_1_jpg.rf.dbce0c0d8c1ba7ade16c84c8e6c4ea62.jpg    765    1020      4   

   xmin  ymin  xmax  ymax  
0   120     0   275   196  
1   202   160   549   758  
2   229   308   455   687  
3   176   330   535   971  
4   204   240   578   790  

Dataset loaded:
X shape: (2248, 1024)
y shape: (2248,)


In [125]:
# s obzirom da su pikseli već normalizovani, možda i nije bilo potrebe za scalerom
model = Pipeline([
    ("scaler", StandardScaler()),

    ("kpca", KernelPCA(
        n_components=100,
        kernel="rbf",
        gamma=0.0005
    )),

    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
    ))
])

In [126]:
model.fit(X, y)

# ============================================================
# evaluacija modela na test skupu
# ============================================================

TEST_DIR = "../../test"
csv_files = [f for f in os.listdir(TEST_DIR) if f.endswith(".csv")]

if len(csv_files) == 0:
    raise Exception("No CSV files found in train folder.")

# objedinjavanje svih .csv fajlova u slučaju da ih je više
dfs = []
for csv_file in csv_files:
    path = os.path.join(TEST_DIR, csv_file)
    dfs.append(pd.read_csv(path))

labels_df = pd.concat(dfs, ignore_index=True)

print("Loaded labels:")
print(labels_df.head())

X_test = []
y_test = []

# učitavanje slika na isti način kao i trening skupu
for _, row in labels_df.iterrows():

    filename = row[0]   
    label = row[3]      

    image_path = os.path.join(TEST_DIR, filename)

    if not os.path.exists(image_path):
        print(f"Missing image: {filename}")
        continue

    img = cv2.imread(image_path)

    if img is None:
        print(f"Failed to load: {filename}")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = cv2.resize(img, IMAGE_SIZE)

    img = img.astype(np.float32) / 255.0

    img = img.flatten()

    X_test.append(img)
    y_test.append(label)

X_test
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Loaded labels:
                                            filename  width  height  class  \
0   4_87_jpg.rf.0c20b4dd0018c701ee1e52e89c2d98fc.jpg    320     240      4   
1    3_5_jpg.rf.0e2afad8283f2f06c202c4e255020750.jpg    764     764      3   
2  3_169_jpg.rf.0b35b78a30d79c430b51966dc0ea2db9.jpg    765    1020      3   
3  5_184_jpg.rf.015d94ca15eaec3e82b12a8275b13cd1.jpg    320     240      5   
4   2_78_jpg.rf.1233620f612e3fd417771ce2d24ee27e.jpg    764     764      2   

   xmin  ymin  xmax  ymax  
0    40    11   170   204  
1   206   193   543   706  
2   222   218   608   768  
3    61     0   236   214  
4   220   131   537   731  

Accuracy: 0.7025

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.71      0.64      0.68        70
           3       0.67      0.66      0.66        82
           4       0.77      0.67  

/home/marko/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/marko/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/marko/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [127]:
#evaluacija na nevidjenim podacima

def predict_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        raise Exception("Could not load image.")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, IMAGE_SIZE)

    img = img.astype(np.float32) / 255.0
    img = img.flatten()

    pred = model.predict([img])[0]

    return pred

In [128]:
# 4 prsta
predict_image('../../my_pictures/mako_4.jpg')

1

In [129]:
# 3 prsta
predict_image('../../my_pictures/Lazarevic_3_srpski.jpg')

1

In [130]:
# 5 prstiju
predict_image('../../my_pictures/Lazarevic_5.jpg')

0